# 08 — Lip Sync

**Purpose:** Warp mouth movements in the source video to match the dubbed audio.

## Model selection rationale

| Model | Architecture | Year | Why included | GPU VRAM |
|---|---|---|---|---|
| **Wav2Lip** | GAN + SyncNet discriminator | 2020 | Baseline; fast; well-tested; easy setup | ~2 GB |
| **LatentSync** (ByteDance) | Latent diffusion on video tokens | 2024 | OSS SOTA; best identity preservation | ~8 GB |
| **MuseTalk 1.5** (Tencent) | Diffusion talking-head | 2024 | Real-time 30 fps on GPU; good quality/speed | ~6 GB |
| **Hallo2** (Fudan) | Hierarchical audio-driven diffusion | 2024 | Best on long video; strong on Indian faces | ~10 GB |

## Why these metrics?
- **LSE-D** = Lip Sync Error Distance via SyncNet. Measures AV distance. Lower = better sync. OTT target: < 7.0. Theatrical: < 4.0.
- **LSE-C** = SyncNet confidence that audio and video are synchronized. Higher = better.
- **ArcFace cosine** = face identity similarity between source and output. Target > 0.85. Below 0.70 = noticeable identity shift — viewers notice the actor's face looks different.
- **Subjective mouth shape (1-5)** = does it look like the actor is saying those words? Objective metrics miss errors like open/closed mouth mismatches on consonants.

## Per-shot ensemble strategy
For long videos, the best approach is per-shot selection: run multiple models on each shot and pick the one with the lowest LSE-D. This is implemented in the ensemble cell at the end.

**Input:** `assembled/final_dubbed.mp4`  
**Output:** `results/final_output.mp4`


In [ ]:
import sys, os, json, time, subprocess, shutil
os.environ['PATH'] = '/opt/homebrew/bin:' + os.environ.get('PATH', '')
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import Video, display

sys.path.insert(0, os.path.abspath('..'))
from config import ASSEMBLED_DIR, LIP_SYNCED_DIR, RESULTS_DIR, SOURCE_VIDEO_PATH

INPUT_VIDEO  = os.path.join(ASSEMBLED_DIR, 'final_dubbed.mp4')
INPUT_AUDIO  = os.path.join(ASSEMBLED_DIR, 'dubbed_audio.wav')
FINAL_OUTPUT = os.path.join(RESULTS_DIR,   'final_output.mp4')

print(f'Input video : {INPUT_VIDEO}  exists={os.path.exists(INPUT_VIDEO)}')
print(f'Input audio : {INPUT_AUDIO}  exists={os.path.exists(INPUT_AUDIO)}')

all_results = {}   # model -> output_path


In [ ]:
try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'GPU: {name}  VRAM: {vram:.1f} GB')
        if vram < 8:
            print('WARNING: < 8 GB VRAM. LatentSync and Hallo2 will OOM. Use Wav2Lip or MuseTalk.')
    else:
        print('No GPU. Wav2Lip only on CPU (slow).')
except ImportError:
    print('PyTorch not installed.')


## Pre-check: face detection scan

All lip sync models produce artefacts on frames with no detected face. Scan first to find those timestamps.


In [ ]:
try:
    import cv2
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    cap   = cv2.VideoCapture(INPUT_VIDEO)
    fps   = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, int(fps))  # 1 sample per second
    no_face_ts = []
    for fi in tqdm(range(0, total, step), desc='Face scan'):
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret: break
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = cascade.detectMultiScale(gray, 1.1, 4)
        if len(faces) == 0:
            no_face_ts.append(round(fi/fps, 1))
    cap.release()
    if no_face_ts:
        print(f'No face at {len(no_face_ts)} timestamps (s): {no_face_ts[:20]}')
    else:
        print(f'Face detected in all {total//step} sampled frames. Good.')
except Exception as e:
    print(f'Face scan skipped: {e}')


## Model 1 — Wav2Lip (GAN baseline, 2020)

Uses SyncNet as a discriminator during training, so the GAN learns audio-visual sync directly. Fast and reliable baseline. Slight lower-face blurring is the main quality issue. [GitHub + weights](https://github.com/Rudrabha/Wav2Lip#getting-the-weights)


In [ ]:
WAV2LIP_DIR = os.path.expanduser('~/Wav2Lip')
CKPT_PATH   = os.path.join(WAV2LIP_DIR, 'checkpoints', 'wav2lip_gan.pth')

if not os.path.exists(WAV2LIP_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/Rudrabha/Wav2Lip.git', WAV2LIP_DIR], check=True)
    print('Cloned. Download wav2lip_gan.pth from repo README.')

print(f'Dir       : {WAV2LIP_DIR}')
print(f'Checkpoint: {CKPT_PATH}  exists={os.path.exists(CKPT_PATH)}')


In [ ]:
WAV2LIP_OUT = os.path.join(LIP_SYNCED_DIR, 'wav2lip_output.mp4')

if os.path.exists(WAV2LIP_OUT):
    all_results['Wav2Lip'] = WAV2LIP_OUT
    print(f'Cached: {WAV2LIP_OUT}')
elif os.path.exists(CKPT_PATH):
    t0 = time.time()
    res = subprocess.run([
        'python', os.path.join(WAV2LIP_DIR, 'inference.py'),
        '--checkpoint_path', CKPT_PATH,
        '--face', INPUT_VIDEO, '--audio', INPUT_AUDIO,
        '--outfile', WAV2LIP_OUT, '--resize_factor', '1', '--pads', '0', '10', '0', '0',
    ], capture_output=True, text=True, cwd=WAV2LIP_DIR)
    if res.returncode != 0:
        print(f'Wav2Lip ERROR:\n{res.stderr[-2000:]}')
    else:
        all_results['Wav2Lip'] = WAV2LIP_OUT
        print(f'Wav2Lip: {time.time()-t0:.0f}s -> {WAV2LIP_OUT}')
else:
    print('Checkpoint not found. Download from: https://github.com/Rudrabha/Wav2Lip#getting-the-weights')


## Model 2 — LatentSync (ByteDance, 2024)

Operates in the latent space of a video VAE — avoids the blurring artefacts of pixel-space GANs. Best identity preservation of the OSS options. Requires ~8 GB VRAM. [GitHub](https://github.com/bytedance/LatentSync)


In [ ]:
LATENTSYNC_DIR  = os.path.expanduser('~/LatentSync')
LATENTSYNC_CKPT = os.path.join(LATENTSYNC_DIR, 'checkpoints', 'latentsync_unet.pt')

if not os.path.exists(LATENTSYNC_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/bytedance/LatentSync.git', LATENTSYNC_DIR], check=True)
    print('Cloned. Download weights from HuggingFace per README.')

print(f'Dir        : {LATENTSYNC_DIR}')
print(f'Checkpoint : exists={os.path.exists(LATENTSYNC_CKPT)}')


In [ ]:
LATENTSYNC_OUT = os.path.join(LIP_SYNCED_DIR, 'latentsync_output.mp4')

if os.path.exists(LATENTSYNC_OUT):
    all_results['LatentSync'] = LATENTSYNC_OUT
    print(f'Cached: {LATENTSYNC_OUT}')
elif os.path.exists(LATENTSYNC_CKPT):
    t0 = time.time()
    res = subprocess.run([
        'python', 'scripts/inference.py',
        '--unet_config_path', 'configs/unet/second_stage.yaml',
        '--inference_ckpt_path', LATENTSYNC_CKPT,
        '--video_path', INPUT_VIDEO, '--audio_path', INPUT_AUDIO,
        '--video_out_path', LATENTSYNC_OUT,
    ], capture_output=True, text=True, cwd=LATENTSYNC_DIR)
    if res.returncode != 0:
        print(f'LatentSync ERROR:\n{res.stderr[-2000:]}')
    else:
        all_results['LatentSync'] = LATENTSYNC_OUT
        print(f'LatentSync: {time.time()-t0:.0f}s -> {LATENTSYNC_OUT}')
else:
    print('Checkpoint not found. See: https://github.com/bytedance/LatentSync')


## Model 3 — MuseTalk 1.5 (Tencent, 2024)

Real-time (30 fps on a 3090). Handles head pose variation better than Wav2Lip. [GitHub](https://github.com/TMElyralab/MuseTalk)


In [ ]:
MUSETALK_DIR = os.path.expanduser('~/MuseTalk')

if not os.path.exists(MUSETALK_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/TMElyralab/MuseTalk.git', MUSETALK_DIR], check=True)
    print('Cloned. Follow README for weights.')

MUSETALK_OUT = os.path.join(LIP_SYNCED_DIR, 'musetalk_output.mp4')

if os.path.exists(MUSETALK_OUT):
    all_results['MuseTalk'] = MUSETALK_OUT
    print(f'Cached: {MUSETALK_OUT}')
else:
    video_name = os.path.splitext(os.path.basename(INPUT_VIDEO))[0]
    res = subprocess.run([
        'python', '-m', 'scripts.inference',
        '--video_path', INPUT_VIDEO, '--audio_path', INPUT_AUDIO,
        '--result_dir', os.path.join(MUSETALK_DIR, 'results'),
    ], capture_output=True, text=True, cwd=MUSETALK_DIR)
    if res.returncode != 0:
        print(f'MuseTalk ERROR:\n{res.stderr[-2000:]}')
    else:
        # FIX: search result dir robustly for the output file
        result_base = os.path.join(MUSETALK_DIR, 'results', video_name)
        found = False
        for search_dir in [result_base, os.path.join(MUSETALK_DIR, 'results')]:
            if not os.path.exists(search_dir): continue
            for fn in os.listdir(search_dir):
                if fn.endswith('.mp4'):
                    shutil.copy2(os.path.join(search_dir, fn), MUSETALK_OUT)
                    all_results['MuseTalk'] = MUSETALK_OUT
                    print(f'MuseTalk -> {MUSETALK_OUT} (from {fn})')
                    found = True
                    break
            if found: break
        if not found:
            print('MuseTalk ran but output not located. Check results dir manually.')


## Model 4 — Hallo2 (Fudan University, 2024)

Hierarchical audio-driven diffusion. Best for long-form video — maintains temporal consistency across many seconds. Strong on non-Western faces. Requires ~10 GB VRAM. [GitHub](https://github.com/fudan-generative-vision/hallo2)


In [ ]:
HALLO2_DIR = os.path.expanduser('~/hallo2')

if not os.path.exists(HALLO2_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/fudan-generative-vision/hallo2.git', HALLO2_DIR], check=True)
    print('Cloned Hallo2. Install deps and download weights per README.')

HALLO2_OUT = os.path.join(LIP_SYNCED_DIR, 'hallo2_output.mp4')
if os.path.exists(HALLO2_OUT):
    all_results['Hallo2'] = HALLO2_OUT
    print(f'Cached: {HALLO2_OUT}')
else:
    # Hallo2 takes a portrait image + audio; extract first frame for test
    test_frame = os.path.join(LIP_SYNCED_DIR, 'hallo2_source_frame.jpg')
    subprocess.run(['ffmpeg', '-y', '-i', INPUT_VIDEO, '-vframes', '1', test_frame],
                   capture_output=True)
    print(f'Hallo2 source frame: {test_frame}')
    print('Run Hallo2 inference manually per its README (portrait + audio format).')
    print('Once done: all_results["Hallo2"] = "path/to/output.mp4"')


## Commercial — Sync.so Lipsync-2 & VEED via fal.ai

Production-grade lip sync APIs. Both typically outperform OSS alternatives on real-world video — less blurring, better identity preservation, no CUDA setup required.

| API | Price | Strengths |
|---|---|---|
| **Sync.so Lipsync-2** | Free tier + $0.10/min | Current commercial SOTA; full video upload |
| **VEED via fal.ai** | ~$0.40/min | Fast; runs existing VEED model on fal.ai GPU cloud |

Enter API keys when prompted or press Enter to skip.

In [ ]:
import requests

SYNCSO_API_KEY = os.getenv('SYNCSO_API_KEY') or getpass('Sync.so API key (Enter to skip): ')
SYNCSO_OUT     = os.path.join(LIP_SYNCED_DIR, 'syncso_output.mp4')

if not SYNCSO_API_KEY.strip():
    print('[Sync.so] Skipped (no API key). Set SYNCSO_API_KEY env var to test later.')

elif os.path.exists(SYNCSO_OUT):
    all_results['Sync.so-Lipsync2'] = SYNCSO_OUT
    print(f'[Sync.so] Cached: {SYNCSO_OUT}')

else:
    try:
        t0 = time.time()

        # Submit job via direct file upload
        with open(INPUT_VIDEO, 'rb') as vf, open(INPUT_AUDIO, 'rb') as af:
            submit_resp = requests.post(
                'https://api.sync.so/lipsync',
                headers={'x-api-key': SYNCSO_API_KEY},
                files={
                    'video': (os.path.basename(INPUT_VIDEO), vf, 'video/mp4'),
                    'audio': (os.path.basename(INPUT_AUDIO), af, 'audio/wav'),
                },
                data={'model': 'lipsync-2', 'synergize': 'true'},
            )
        submit_resp.raise_for_status()
        job_id = submit_resp.json().get('id') or submit_resp.json().get('job_id')
        print(f'Sync.so job: {job_id}. Polling...')

        # Poll for completion
        while True:
            time.sleep(15)
            status_resp = requests.get(
                f'https://api.sync.so/lipsync/{job_id}',
                headers={'x-api-key': SYNCSO_API_KEY},
            )
            status = status_resp.json().get('status', 'unknown')
            print(f'  Status: {status}')
            if status == 'completed': break
            if status in ('error', 'failed'):
                raise RuntimeError(f'Sync.so job failed: {status_resp.json()}')

        # Download result
        output_url = status_resp.json().get('output_url') or status_resp.json().get('url')
        dl_resp = requests.get(output_url)
        with open(SYNCSO_OUT, 'wb') as f:
            f.write(dl_resp.content)

        elapsed = time.time() - t0
        all_results['Sync.so-Lipsync2'] = SYNCSO_OUT
        print(f'Sync.so: {elapsed:.0f}s -> {SYNCSO_OUT}  ({os.path.getsize(SYNCSO_OUT)/1e6:.1f} MB)')
    except Exception as e:
        print(f'Sync.so FAILED: {e}')

In [ ]:
# ── VEED via fal.ai ───────────────────────────────────────────────────────────
# pip install fal-client
# fal.ai runs VEED's lip sync model on their GPU cloud. ~$0.40/min.
# Sign up at fal.ai; no CC required for small usage with free credits.
FAL_KEY   = os.getenv('FAL_KEY') or getpass('fal.ai API key (Enter to skip): ')
VEED_OUT  = os.path.join(LIP_SYNCED_DIR, 'veed_output.mp4')

if not FAL_KEY.strip():
    print('[VEED/fal.ai] Skipped (no API key). Set FAL_KEY env var to test later.')

elif os.path.exists(VEED_OUT):
    all_results['VEED-fal.ai'] = VEED_OUT
    print(f'[VEED/fal.ai] Cached: {VEED_OUT}')

else:
    try:
        import fal_client
        os.environ['FAL_KEY'] = FAL_KEY

        t0 = time.time()

        # Upload video and audio to fal.ai storage
        print('Uploading video and audio to fal.ai...')
        with open(INPUT_VIDEO, 'rb') as vf:
            video_url = fal_client.upload(vf, 'video/mp4')
        with open(INPUT_AUDIO, 'rb') as af:
            audio_url = fal_client.upload(af, 'audio/wav')
        print(f'  video: {video_url}')
        print(f'  audio: {audio_url}')

        result = fal_client.subscribe(
            'fal-ai/veed/lipsync',
            arguments={'video_url': video_url, 'audio_url': audio_url},
            with_logs=True,
        )

        output_url = result['video']['url']
        dl_resp = requests.get(output_url)
        with open(VEED_OUT, 'wb') as f:
            f.write(dl_resp.content)

        elapsed = time.time() - t0
        all_results['VEED-fal.ai'] = VEED_OUT
        print(f'VEED/fal.ai: {elapsed:.0f}s -> {VEED_OUT}  ({os.path.getsize(VEED_OUT)/1e6:.1f} MB)')
    except ImportError:
        print('fal-client not installed. Run: pip install fal-client')
    except Exception as e:
        print(f'VEED/fal.ai FAILED: {e}')

## Metrics

**SyncNet setup:** `git clone https://github.com/joonson/syncnet_python.git ~/syncnet_python`  
**ArcFace setup:** `pip install insightface onnxruntime`


In [ ]:
import re as _re

SYNCNET_DIR    = os.path.expanduser('~/syncnet_python')
syncnet_scores = {}

if not os.path.exists(SYNCNET_DIR):
    print('SyncNet not found.')
    print('Install: git clone https://github.com/joonson/syncnet_python.git ~/syncnet_python')
else:
    for name, vid_path in tqdm(all_results.items(), desc='SyncNet'):
        try:
            res = subprocess.run(
                ['python', 'run_pipeline.py', '--videofile', os.path.abspath(vid_path),
                 '--reference', name, '--data_dir', '/tmp/syncnet_tmp'],
                capture_output=True, text=True, cwd=SYNCNET_DIR
            )
            lse_d = lse_c = None
            for line in res.stdout.splitlines() + res.stderr.splitlines():
                m = _re.search(r'Min dist\s*[:\s]+([\d.]+)', line)
                if m: lse_d = float(m.group(1))
                m = _re.search(r'Confidence\s*[:\s]+([\d.]+)', line)
                if m: lse_c = float(m.group(1))
            syncnet_scores[name] = {'lse_d': lse_d, 'lse_c': lse_c}
            print(f'  {name}: LSE-D={lse_d}  LSE-C={lse_c}')
        except Exception as e:
            syncnet_scores[name] = {'lse_d': None, 'lse_c': None}
            print(f'  {name}: SyncNet failed: {e}')


In [ ]:
import numpy as np   # explicit import — required in this cell scope
import cv2
from numpy.linalg import norm

arcface_scores = {}
try:
    from insightface.app import FaceAnalysis

    app_af = FaceAnalysis(name='buffalo_l',
                          providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    app_af.prepare(ctx_id=0)

    def sample_embeddings(path, n=20):
        cap   = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step  = max(1, total // n)
        embs  = []
        for fi in range(0, total, step):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if not ret: break
            faces = app_af.get(frame)
            if faces: embs.append(faces[0].normed_embedding)
        cap.release()
        return embs

    src_embs = sample_embeddings(SOURCE_VIDEO_PATH)
    if src_embs:
        src_mean = np.mean(src_embs, axis=0)
        src_mean /= norm(src_mean)
        for name, vid_path in tqdm(all_results.items(), desc='ArcFace'):
            out_embs = sample_embeddings(vid_path)
            if out_embs:
                out_mean  = np.mean(out_embs, axis=0)
                out_mean /= norm(out_mean)
                cos = float(np.dot(src_mean, out_mean))
                arcface_scores[name] = round(cos, 4)
                quality = 'excellent' if cos>0.85 else ('ok' if cos>0.70 else 'poor')
                print(f'  {name}: {cos:.4f}  ({quality})')
            else:
                arcface_scores[name] = None
    else:
        print('No faces in source video.')
except ImportError:
    print('InsightFace not installed: pip install insightface onnxruntime')
except Exception as e:
    print(f'ArcFace failed: {e}')


In [ ]:
# Fill in after watching each output. 1=bad, 5=excellent.
subjective = {
    'Wav2Lip':           {'mouth_accuracy': None, 'identity_preserved': None, 'overall': None},
    'LatentSync':        {'mouth_accuracy': None, 'identity_preserved': None, 'overall': None},
    'MuseTalk':          {'mouth_accuracy': None, 'identity_preserved': None, 'overall': None},
    'Hallo2':            {'mouth_accuracy': None, 'identity_preserved': None, 'overall': None},
    'Sync.so-Lipsync2':  {'mouth_accuracy': None, 'identity_preserved': None, 'overall': None},
    'VEED-fal.ai':       {'mouth_accuracy': None, 'identity_preserved': None, 'overall': None},
}
for name, s in subjective.items():
    print(f'{name:18s}  mouth={s["mouth_accuracy"]}  identity={s["identity_preserved"]}  overall={s["overall"]}')

In [ ]:
rows = []
for name in all_results:
    sc = syncnet_scores.get(name, {})
    rows.append({
        'Model':           name,
        'LSE-D':           sc.get('lse_d'),
        'LSE-C':           sc.get('lse_c'),
        'ArcFace':         arcface_scores.get(name),
        'Mouth (subj)':    subjective.get(name, {}).get('mouth_accuracy'),
        'Identity (subj)': subjective.get(name, {}).get('identity_preserved'),
        'Overall (subj)':  subjective.get(name, {}).get('overall'),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))


In [ ]:
# ── Select winner: lowest LSE-D if available, else model preference ────────────
if syncnet_scores and any(v.get('lse_d') is not None for v in syncnet_scores.values()):
    valid  = {k: v['lse_d'] for k,v in syncnet_scores.items() if v.get('lse_d') is not None}
    WINNER = min(valid, key=valid.get)
    print(f'Auto winner by LSE-D: {WINNER}  (LSE-D={valid[WINNER]:.2f})')
elif all_results:
    for pref in ['LatentSync', 'Hallo2', 'MuseTalk', 'Wav2Lip']:
        if pref in all_results:
            WINNER = pref
            break
    else:
        WINNER = list(all_results.keys())[0]
    print(f'Auto winner by preference: {WINNER}')
else:
    WINNER = None

# Manual override:
# WINNER = 'Wav2Lip'

if WINNER and WINNER in all_results:
    shutil.copy2(all_results[WINNER], FINAL_OUTPUT)
else:
    shutil.copy2(os.path.join(ASSEMBLED_DIR, 'final_dubbed.mp4'), FINAL_OUTPUT)
    print('No lip sync ran — using audio-only dubbed video as final output.')

print(f'\n{"="*60}\nPIPELINE COMPLETE\n{"="*60}')
print(f'Winner      : {WINNER}')
print(f'Final output: {FINAL_OUTPUT}')
print(f'Exists      : {os.path.exists(FINAL_OUTPUT)}')
if os.path.exists(FINAL_OUTPUT):
    print(f'Size        : {os.path.getsize(FINAL_OUTPUT)/1e6:.1f} MB')
